In [1]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
from datetime import datetime, timezone

_root = Path("..").resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# Load .env from project root (contains BLOCKFROST_PROJECT_ID)
from dotenv import load_dotenv
load_dotenv(dotenv_path=_root / ".env")

from src.feature_engineering import RAW_DIR, ENG_DIR, engineer_features, save_engineered

from coinmetrics.api_client import CoinMetricsClient  # pip install coinmetrics-api-client

In [2]:
# Initialize the client (Community/Free tier doesn't require an API key for many basic calls)
client = CoinMetricsClient()

In [3]:
import requests
from blockfrost import BlockFrostApi

# Metrics per asset — BTC gets exchange flows, hash rate, and issuance;
# ADA gets MVRV and SplyCur (needed to compute StakingRatio).
ASSET_METRICS: dict[str, list[str]] = {
    "btc": [
        "PriceUSD", "CapMrktCurUSD", "TxCnt", "AdrActCnt",
        "CapMVRVCur", "HashRate",
        "FlowInExUSD", "FlowOutExUSD", "IssTotUSD", "FeeTotNtv",
    ],
    "ada": [
        "PriceUSD", "CapMrktCurUSD", "TxCnt", "AdrActCnt",
        "CapMVRVCur", "SplyCur",
    ],
}


def fetch_fear_and_greed() -> pd.DataFrame:
    """Fetch full Fear & Greed history from Alternative.me (no API key required).

    Returns a DataFrame with a DatetimeIndex and column ``FearGreedValue`` (0-100).
    History starts 2018-02-01.
    """
    url = "https://api.alternative.me/fng/?limit=0&date_format=world"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    entries = resp.json()["data"]
    df = pd.DataFrame(entries)
    df["date"] = pd.to_datetime(df["timestamp"], format="%d-%m-%Y")
    df = (
        df.rename(columns={"value": "FearGreedValue"})[["date", "FearGreedValue"]]
        .assign(FearGreedValue=lambda x: pd.to_numeric(x["FearGreedValue"]))
        .set_index("date")
        .sort_index()
    )
    print(f"Fear & Greed: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
    return df


def fetch_blockfrost_ada_staking(start_epoch: int = 208) -> pd.DataFrame:
    """Fetch per-epoch ADA staking data from Blockfrost, expanded to daily frequency.

    Shelley (staking) started at epoch 208 (2020-07-29). Earlier dates get NaN.
    Returns a DataFrame with DatetimeIndex and column ``ActiveStakeADA`` (in ADA).

    Requires BLOCKFROST_PROJECT_ID in the environment (loaded from .env).
    """
    project_id = os.environ.get("BLOCKFROST_PROJECT_ID", "")
    if not project_id or "XXXX" in project_id:
        raise ValueError("BLOCKFROST_PROJECT_ID not set in .env")

    api = BlockFrostApi(project_id=project_id)
    latest_epoch = api.epoch_latest().epoch
    total = latest_epoch - start_epoch + 1
    print(f"Blockfrost: fetching {total} epochs ({start_epoch} to {latest_epoch})...")

    records = []
    for i, ep in enumerate(range(start_epoch, latest_epoch + 1)):
        try:
            e = api.epoch(ep)
            stake_ada = int(e.active_stake) / 1e6 if e.active_stake else None
            # Normalize to midnight so the index aligns with the daily Coin Metrics index
            records.append({
                "epoch_start": pd.Timestamp(e.start_time, unit="s").normalize(),
                "epoch_end":   pd.Timestamp(e.end_time,   unit="s").normalize(),
                "ActiveStakeADA": stake_ada,
            })
        except Exception as ex:
            print(f"  Epoch {ep} skipped: {ex}")
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{total} epochs done...")

    if not records:
        raise RuntimeError("No epoch data returned by Blockfrost")

    edf = pd.DataFrame(records).sort_values("epoch_start")

    # Expand each 5-day epoch into daily rows (forward-fill within epoch)
    daily_rows = []
    for _, row in edf.iterrows():
        day = row["epoch_start"]
        while day < row["epoch_end"]:
            daily_rows.append({"date": day, "ActiveStakeADA": row["ActiveStakeADA"]})
            day += pd.Timedelta(days=1)

    daily_df = (
        pd.DataFrame(daily_rows)
        .set_index("date")
        .sort_index()
    )
    daily_df.index = pd.to_datetime(daily_df.index)
    print(f"Blockfrost: {len(edf)} epochs expanded to {len(daily_df)} daily rows")
    return daily_df

In [4]:
def fetch_and_save_crypto_data(asset_ticker, fg_df=None, extra_df=None, start='2020-01-01', end='2026-06-01'):
    """Fetch Coin Metrics metrics for an asset and save the raw CSV.

    Parameters
    ----------
    asset_ticker : str
    fg_df : pd.DataFrame or None
        Fear & Greed DataFrame (DatetimeIndex, column FearGreedValue).
    extra_df : pd.DataFrame or None
        Additional daily DataFrame to merge (e.g. Blockfrost staking for ADA).
    start, end : str  ISO date strings
    """
    asset_ticker = asset_ticker.lower()
    metrics = ASSET_METRICS.get(asset_ticker, ASSET_METRICS["ada"])

    print(f"\n--- Processing {asset_ticker.upper()} ---")
    print(f"Fetching {len(metrics)} metrics: {metrics}")

    try:
        data = client.get_asset_metrics(
            assets=asset_ticker,
            metrics=metrics,
            start_time=start,
            end_time=end,
            frequency='1d'
        )
        df = data.to_dataframe()

        # Normalise index to timezone-naive date (Coin Metrics returns UTC timestamps)
        df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None).dt.normalize()
        df = df.set_index('time')
        df.index.name = 'time'

        # Drop Coin Metrics metadata columns (e.g. "FlowInExUSD-status", "-status-time")
        status_cols = [c for c in df.columns if '-status' in c]
        if status_cols:
            df = df.drop(columns=status_cols)

        for col in metrics:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Derived metrics computed here (use raw columns before any drop)
        df['NVT_Tx_Basis'] = df['CapMrktCurUSD'] / df['TxCnt'].rolling(window=28).mean()
        df['Activity_Velocity'] = df['TxCnt'] / df['AdrActCnt']

        # Merge Fear & Greed
        if fg_df is not None:
            df = df.join(fg_df, how='left')
            n_missing = df['FearGreedValue'].isna().sum()
            if n_missing:
                print(f"  FearGreedValue: {n_missing} rows before F&G history (expected)")

        # Merge any extra source (e.g. Blockfrost staking for ADA)
        if extra_df is not None:
            df = df.join(extra_df, how='left')
            for col in extra_df.columns:
                n_missing = df[col].isna().sum()
                if n_missing:
                    print(f"  {col}: {n_missing} NaN rows (pre-Shelley or coverage gap)")

        # Save raw
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        file_path = RAW_DIR / f'coinmetrics_{asset_ticker}.csv'
        df.to_csv(file_path)

        print(f"SUCCESS: {len(df)} rows x {len(df.columns)} columns -> {file_path}")
        print(f"  Columns: {list(df.columns)}")
        return df

    except Exception as e:
        print(f"ERROR fetching {asset_ticker.upper()}: {e}")
        return None

### Feature engineering & disk layout

All feature transformations and the on-disk paths live in `src/feature_engineering.py`:

- `engineer_features(df)` — lags (1/3/7/14/30d on `Tx_Intensity`, `AdrActCnt`, `NVT_Tx_Basis`, `Velocity_Momentum`), price-distance lags (30/90/182d), `is_weekend`, `Price_vs_MA7`, `Volatility_30d`, and forward-return targets `fwd_return_{3,7,14,30}d`.
- `save_engineered(df, asset)` — writes both `data/engineered/engineered_{asset}.csv` and `.parquet`.
- `load_engineered_frames()` (used by the model notebooks) will rebuild engineered files from `data/raw/coinmetrics_{asset}.csv` on demand if the cache is missing.

In [5]:
# engineer_features and save_engineered are imported in the first cell from
# src.feature_engineering. The final cell of this notebook orchestrates the
# fetch + engineer + save pipeline for each asset.

```mermaid
flowchart TD
    A[Start: fetch_and_save_crypto_data asset] --> C[CoinMetricsClient.get_asset_metrics]
    C --> D[to_dataframe + datetime index + numeric coercion]
    D --> F[Add NVT_Tx_Basis and Activity_Velocity in this notebook]
    F --> G[Write raw CSV to data/raw/coinmetrics_asset.csv]
    G --> H[engineer_features in src/feature_engineering.py]
    H --> I[save_engineered: data/engineered/engineered_asset.csv and .parquet]
    C -->|exception| J[Print error and return None]
```

In [6]:
%%time
if __name__ == "__main__":
    fg_df = fetch_fear_and_greed()
    ada_staking_df = fetch_blockfrost_ada_staking(start_epoch=208)

    assets_to_fetch = ['btc', 'ada']
    for asset in assets_to_fetch:
        # ADA gets staking data merged; BTC gets None (ignored inside the function)
        extra = ada_staking_df if asset == 'ada' else None
        raw_df = fetch_and_save_crypto_data(asset, fg_df=fg_df, extra_df=extra)
        if raw_df is None:
            continue
        engineered = engineer_features(raw_df)
        save_engineered(engineered, asset)

Fear & Greed: 3055 days (2018-02-01 to 2026-06-17)
Blockfrost: fetching 430 epochs (208 to 637)...
  100/430 epochs done...
  200/430 epochs done...
  300/430 epochs done...
  400/430 epochs done...
Blockfrost: 430 epochs expanded to 2150 daily rows

--- Processing BTC ---
Fetching 10 metrics: ['PriceUSD', 'CapMrktCurUSD', 'TxCnt', 'AdrActCnt', 'CapMVRVCur', 'HashRate', 'FlowInExUSD', 'FlowOutExUSD', 'IssTotUSD', 'FeeTotNtv']
  FearGreedValue: 1 rows before F&G history (expected)
SUCCESS: 2344 rows x 14 columns -> C:\Users\gabri\OneDrive\Universidades\XPe\IC\data\raw\coinmetrics_btc.csv
  Columns: ['asset', 'PriceUSD', 'CapMrktCurUSD', 'TxCnt', 'AdrActCnt', 'CapMVRVCur', 'HashRate', 'FlowInExUSD', 'FlowOutExUSD', 'IssTotUSD', 'FeeTotNtv', 'NVT_Tx_Basis', 'Activity_Velocity', 'FearGreedValue']
Engineered BTC saved to:
  - C:\Users\gabri\OneDrive\Universidades\XPe\IC\data\engineered\engineered_btc.csv
  - C:\Users\gabri\OneDrive\Universidades\XPe\IC\data\engineered\engineered_btc.parquet